In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import pandas as pd
import numpy as np
import time
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 경로 설정 (본인 환경에 맞게 수정)
base_path = '/content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘'
# 파일이 없으면 에러가 나므로 예외처리 혹은 경로 확인 필요
ratings_path = os.path.join(base_path, 'ratings.csv')
movies_path = os.path.join(base_path, 'movies.csv')

# ---------------------------------------------------------
# 1. 데이터 로드
# ---------------------------------------------------------
try:
    ratings = pd.read_csv(ratings_path)
    movies  = pd.read_csv(movies_path)
    print(f"데이터 로드 완료: Ratings {ratings.shape}, Movies {movies.shape}")
except FileNotFoundError:
    print(f"파일을 찾을 수 없습니다. 경로를 확인해주세요: {base_path}")
    # 실행 테스트를 위한 더미 데이터 생성 (파일 없을 시)
    ratings = pd.DataFrame(columns=['userId', 'movieId', 'rating'])
    movies = pd.DataFrame(columns=['movieId', 'title'])

SEED_LIST = [10, 21, 35, 42, 57]
TEST_SAMPLE_SIZE = 3000
SHRINKAGE_PARAM = 50
PERSONA_HEAVY_USER = 414       # 헤비 유저 ID
PERSONA_GENRE_SPECIALIST = 85  # 드라마 전문가 ID

데이터 로드 완료: Ratings (100836, 4), Movies (9742, 3)


# [1] weight=50

In [10]:
# 전처리, 모델링, 평가 및 결과 출력

# ---------------------------------------------------------
# 2. 전처리 (Z-Score)
# ---------------------------------------------------------
def run_preprocessing(train_df, test_df):
    train = train_df.copy()
    test  = test_df.copy()

    user_stats = train.groupby('userId')['rating'].agg(['mean', 'std']).fillna(1.0)
    user_stats['std'] = user_stats['std'].replace(0, 1.0)

    train = train.merge(user_stats, on='userId', suffixes=('', '_user'))
    train['z_rating'] = (train['rating'] - train['mean']) / train['std']

    test = test.merge(user_stats, on='userId', how='left')
    test = test.dropna(subset=['mean', 'std'])
    test['z_rating'] = (test['rating'] - test['mean']) / test['std']

    return train, test, user_stats

# ---------------------------------------------------------
# 3. 평가 함수 (Significance Weighting 적용)
# ---------------------------------------------------------
def fast_predict_matrix_shrinkage(train_df, shrinkage=100):
    """
    Significance Weighting을 적용하여 신뢰도 낮은 유사도를 제거/축소
    """
    # 1. Pivot
    pivot_df = train_df.pivot(index='userId', columns='movieId', values='z_rating').fillna(0)
    R = pivot_df.values
    user_ids = pivot_df.index
    movie_ids = pivot_df.columns

    # 2. 기본 코사인 유사도 계산
    item_norms = np.linalg.norm(R.T, axis=1, keepdims=True)
    item_norms = np.where(item_norms == 0, 1.0, item_norms)
    sim_matrix = np.dot(R.T, R) / (item_norms @ item_norms.T)

    # 3. Significance Weighting (De-noising)
    R_binary = (R != 0).astype(float)
    common_counts = np.dot(R_binary.T, R_binary)

    weighting_factor = np.minimum(common_counts, shrinkage) / shrinkage
    sim_matrix_shrunk = sim_matrix * weighting_factor

    # 대각선 0 처리 및 양수 유사도만 사용
    np.fill_diagonal(sim_matrix_shrunk, 0)
    S = np.where(sim_matrix_shrunk > 0, sim_matrix_shrunk, 0)

    # 4. 예측 (Weighted Sum)
    numerator = np.dot(R, S)
    denominator = np.dot(R_binary, np.abs(S))

    with np.errstate(divide='ignore', invalid='ignore'):
        pred_z = numerator / denominator
        valid_mask = (denominator > 0) & np.isfinite(pred_z)
        pred_z = np.where(valid_mask, pred_z, 0.0)

    return pred_z, user_ids, movie_ids, valid_mask

# ---------------------------------------------------------
# 4. 페르소나 추천 함수 (추가됨)
# ---------------------------------------------------------
def get_recommendations(user_id, pred_matrix, u_ids, m_ids, valid_mask, user_stats, train_df, top_k=3):
    if user_id not in u_ids:
        return None

    u_idx = np.where(u_ids == user_id)[0][0]
    user_preds = pred_matrix[u_idx]
    user_valid = valid_mask[u_idx]

    # 이미 본 영화 제외
    seen_movies = set(train_df[train_df['userId'] == user_id]['movieId'])

    candidates = []
    u_mean = user_stats.loc[user_id, 'mean']
    u_std = user_stats.loc[user_id, 'std']

    for m_idx, m_id in enumerate(m_ids):
        if m_id not in seen_movies and user_valid[m_idx]:
            score_z = user_preds[m_idx]
            score_r = score_z * u_std + u_mean
            candidates.append((m_id, score_r))

    # 점수순 정렬
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]

# ---------------------------------------------------------
# 5. 실행 루프 및 결과 출력
# ---------------------------------------------------------
results = []
best_reco = {} # 추천 결과 저장용

print(f"실험 시작: Significance Weighting (Shrinkage={SHRINKAGE_PARAM})")
print(f"설정: Test Sample Size = {TEST_SAMPLE_SIZE}")
print("-" * 60)

for idx, SEED in enumerate(SEED_LIST):
    # 데이터 분할 및 전처리
    train_raw, test_raw = train_test_split(ratings, test_size=0.2, random_state=SEED)
    train, test, user_stats = run_preprocessing(train_raw, test_raw)

    # 행렬 연산 시간 측정
    s_time = time.perf_counter()
    pred_matrix_z, u_ids, m_ids, valid_mask = fast_predict_matrix_shrinkage(train, shrinkage=SHRINKAGE_PARAM)
    e_time = time.perf_counter() - s_time

    u_map = {u: i for i, u in enumerate(u_ids)}
    m_map = {m: i for i, m in enumerate(m_ids)}

    # 테스트 샘플링
    test_sample = test.sample(n=min(TEST_SAMPLE_SIZE, len(test)), random_state=SEED)

    y_true = []
    y_pred = []
    fallback_cnt = 0 # 폴백 카운트 초기화

    for _, row in test_sample.iterrows():
        uid, mid, true_r, u_mean, u_std = row['userId'], row['movieId'], row['rating'], row['mean'], row['std']

        predicted = False
        if uid in u_map and mid in m_map:
            u_idx = u_map[uid]
            m_idx = m_map[mid]

            if valid_mask[u_idx, m_idx]:
                pred_z = pred_matrix_z[u_idx, m_idx]
                pred_r = pred_z * u_std + u_mean
                y_pred.append(pred_r)
                y_true.append(true_r)
                predicted = True

        if not predicted:
            y_pred.append(u_mean)
            y_true.append(true_r)
            fallback_cnt += 1 # 카운트 증가

    # 메트릭 계산
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    mae_val = mean_absolute_error(y_true, y_pred)

    avg_ms = (e_time / len(test_sample)) * 1000 # 샘플당 속도

    # 결과 저장
    results.append({
        'seed': SEED,
        'rmse': rmse_val,
        'mae': mae_val,
        'avg_ms': avg_ms,
        'fallback_count': fallback_cnt,
        'matrix_time': e_time
    })

    # Seed 10일 때 페르소나 추천 수행
    if SEED == 10:
        best_reco['heavy'] = get_recommendations(PERSONA_HEAVY_USER, pred_matrix_z, u_ids, m_ids, valid_mask, user_stats, train)
        best_reco['specialist'] = get_recommendations(PERSONA_GENRE_SPECIALIST, pred_matrix_z, u_ids, m_ids, valid_mask, user_stats, train)

# ---------------------------------------------------------
# 6. 최종 결과 포맷팅
# ---------------------------------------------------------
df_res = pd.DataFrame(results)

print("\n=== 실험 결과 요약 ===")
print(df_res[['seed', 'rmse', 'mae', 'avg_ms', 'fallback_count']].to_string(index=False))

print("\n=== 평균 ± 표준편차 ===")
print(f"RMSE : {df_res['rmse'].mean():.4f} ± {df_res['rmse'].std():.4f}")
print(f"MAE  : {df_res['mae'].mean():.4f} ± {df_res['mae'].std():.4f}")
print(f"폴백 사용 케이스 : {df_res['fallback_count'].mean():.1f}개 (± {df_res['fallback_count'].std():.1f})")
print(f"예측 속도 (샘플 기준) : {df_res['avg_ms'].mean():.2f} ms (± {df_res['avg_ms'].std():.2f})")
print(f"행렬 계산 시간 : {df_res['matrix_time'].mean():.4f} s (± {df_res['matrix_time'].std():.4f})")

print("\n=== 페르소나 추천 결과 (Seed 10) ===")
def print_reco(name, uid, recs):
    print(f"{name} (userId={uid}):", end=" ")
    if recs:
        print()
        for r in recs:
            try:
                title = movies[movies['movieId'] == r[0]]['title'].values[0]
                print(f"  - {title} (예측점수: {r[1]:.2f})")
            except:
                print(f"  - MovieID {r[0]} (Title Not Found)")
    else:
        print("추천 결과 없음")

print_reco("헤비 유저", PERSONA_HEAVY_USER, best_reco.get('heavy'))
print_reco("드라마 전문가", PERSONA_GENRE_SPECIALIST, best_reco.get('specialist'))

실험 시작: Significance Weighting (Shrinkage=50)
설정: Test Sample Size = 3000
------------------------------------------------------------

=== 실험 결과 요약 ===
 seed     rmse      mae   avg_ms  fallback_count
   10 0.850792 0.640598 4.703673             126
   21 0.859766 0.651970 4.247263             117
   35 0.866408 0.658003 4.056990             109
   42 0.878644 0.661152 3.968330             117
   57 0.833263 0.633358 4.210210             121

=== 평균 ± 표준편차 ===
RMSE : 0.8578 ± 0.0171
MAE  : 0.6490 ± 0.0118
폴백 사용 케이스 : 118.0개 (± 6.2)
예측 속도 (샘플 기준) : 4.24 ms (± 0.28)
행렬 계산 시간 : 12.7119 s (± 0.8528)

=== 페르소나 추천 결과 (Seed 10) ===
헤비 유저 (userId=414): 
  - Harmonists, The (1997) (예측점수: 5.00)
  - Taxi 3 (2003) (예측점수: 4.93)
  - Spiral (2018) (예측점수: 4.75)
드라마 전문가 (userId=85): 
  - Mr. Wrong (1996) (예측점수: 5.00)
  - Before and After (1996) (예측점수: 5.00)
  - Awfully Big Adventure, An (1995) (예측점수: 5.00)


# weight에 따른 성능 비교

In [2]:
import pandas as pd
import numpy as np
import time
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ---------------------------------------------------------
# 1. 설정 및 데이터 로드
# ---------------------------------------------------------
base_path = '/content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘'
ratings_path = os.path.join(base_path, 'ratings.csv')
movies_path = os.path.join(base_path, 'movies.csv')

try:
    ratings = pd.read_csv(ratings_path)
    movies  = pd.read_csv(movies_path)
except FileNotFoundError:
    print("파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    ratings = pd.DataFrame(columns=['userId', 'movieId', 'rating'])
    movies = pd.DataFrame(columns=['movieId', 'title'])

# 비교할 Shrinkage Weight 목록
# 0: 가중치 없음 (Baseline), 50: 기존 설정, 100+: 강한 규제
SHRINKAGE_CANDIDATES = [0, 10, 30, 50, 100]

SEED_LIST = [10, 21, 35, 42, 57]
TEST_SAMPLE_SIZE = 3000
PERSONA_HEAVY_USER = 414
PERSONA_GENRE_SPECIALIST = 85

# ---------------------------------------------------------
# 2. 전처리 (Z-Score)
# ---------------------------------------------------------
def run_preprocessing(train_df, test_df):
    train = train_df.copy()
    test  = test_df.copy()

    user_stats = train.groupby('userId')['rating'].agg(['mean', 'std']).fillna(1.0)
    user_stats['std'] = user_stats['std'].replace(0, 1.0)

    train = train.merge(user_stats, on='userId', suffixes=('', '_user'))
    train['z_rating'] = (train['rating'] - train['mean']) / train['std']

    test = test.merge(user_stats, on='userId', how='left')
    test = test.dropna(subset=['mean', 'std'])
    test['z_rating'] = (test['rating'] - test['mean']) / test['std']

    return train, test, user_stats

# ---------------------------------------------------------
# 3. 모델링 함수 (Shrinkage 파라미터 적용)
# ---------------------------------------------------------
def fast_predict_matrix_shrinkage(train_df, shrinkage=0):
    """
    shrinkage=0 이면 기본 코사인 유사도,
    shrinkage>0 이면 공통 지지도 기반 가중치 적용
    """
    pivot_df = train_df.pivot(index='userId', columns='movieId', values='z_rating').fillna(0)
    R = pivot_df.values
    user_ids = pivot_df.index
    movie_ids = pivot_df.columns

    # 1. 기본 코사인 유사도
    item_norms = np.linalg.norm(R.T, axis=1, keepdims=True)
    item_norms = np.where(item_norms == 0, 1.0, item_norms)
    sim_matrix = np.dot(R.T, R) / (item_norms @ item_norms.T)

    # 2. Significance Weighting (De-noising)
    if shrinkage > 0:
        R_binary = (R != 0).astype(float)
        common_counts = np.dot(R_binary.T, R_binary)
        weighting_factor = np.minimum(common_counts, shrinkage) / shrinkage
        sim_matrix_shrunk = sim_matrix * weighting_factor
    else:
        # Shrinkage가 0이면 원본 유사도 그대로 사용
        sim_matrix_shrunk = sim_matrix

    np.fill_diagonal(sim_matrix_shrunk, 0)
    S = np.where(sim_matrix_shrunk > 0, sim_matrix_shrunk, 0)

    # 3. 예측
    numerator = np.dot(R, S)
    denominator = np.dot((R != 0).astype(float), np.abs(S))

    with np.errstate(divide='ignore', invalid='ignore'):
        pred_z = numerator / denominator
        valid_mask = (denominator > 0) & np.isfinite(pred_z)
        pred_z = np.where(valid_mask, pred_z, 0.0)

    return pred_z, user_ids, movie_ids, valid_mask

# ---------------------------------------------------------
# 4. 페르소나 추천 함수
# ---------------------------------------------------------
def get_recommendations(user_id, pred_matrix, u_ids, m_ids, valid_mask, user_stats, train_df, top_k=3):
    if user_id not in u_ids:
        return None

    u_idx = np.where(u_ids == user_id)[0][0]
    user_preds = pred_matrix[u_idx]
    user_valid = valid_mask[u_idx]

    seen_movies = set(train_df[train_df['userId'] == user_id]['movieId'])
    candidates = []
    u_mean = user_stats.loc[user_id, 'mean']
    u_std = user_stats.loc[user_id, 'std']

    for m_idx, m_id in enumerate(m_ids):
        if m_id not in seen_movies and user_valid[m_idx]:
            score_z = user_preds[m_idx]
            score_r = score_z * u_std + u_mean
            candidates.append((m_id, score_r))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_k]

def print_reco(name, uid, recs):
    print(f"{name} (userId={uid}):")
    if recs:
        for r in recs:
            try:
                title = movies[movies['movieId'] == r[0]]['title'].values[0]
                print(f"  - {title} (예측점수: {r[1]:.2f})")
            except:
                print(f"  - MovieID {r[0]} (Title Not Found)")
    else:
        print("  추천 결과 없음")

# ---------------------------------------------------------
# 5. 실험 실행 (Shrinkage Loop)
# ---------------------------------------------------------
global_summary = [] # 파라미터별 최종 결과를 담을 리스트

for shrinkage_val in SHRINKAGE_CANDIDATES:
    print(f"\n{'='*60}")
    print(f" 실험 파라미터: Shrinkage = {shrinkage_val}")
    print(f" (0이면 미적용, 숫자가 클수록 신뢰도 필터링 강도 높음)")
    print(f"{'='*60}")

    results = []
    best_reco = {}

    for idx, SEED in enumerate(SEED_LIST):
        train_raw, test_raw = train_test_split(ratings, test_size=0.2, random_state=SEED)
        train, test, user_stats = run_preprocessing(train_raw, test_raw)

        s_time = time.perf_counter()
        pred_matrix_z, u_ids, m_ids, valid_mask = fast_predict_matrix_shrinkage(train, shrinkage=shrinkage_val)
        e_time = time.perf_counter() - s_time

        u_map = {u: i for i, u in enumerate(u_ids)}
        m_map = {m: i for i, m in enumerate(m_ids)}
        test_sample = test.sample(n=min(TEST_SAMPLE_SIZE, len(test)), random_state=SEED)

        y_true = []
        y_pred = []
        fallback_cnt = 0

        for _, row in test_sample.iterrows():
            uid, mid, true_r, u_mean, u_std = row['userId'], row['movieId'], row['rating'], row['mean'], row['std']

            predicted = False
            if uid in u_map and mid in m_map:
                u_idx, m_idx = u_map[uid], m_map[mid]
                if valid_mask[u_idx, m_idx]:
                    pred_z = pred_matrix_z[u_idx, m_idx]
                    y_pred.append(pred_z * u_std + u_mean)
                    y_true.append(true_r)
                    predicted = True

            if not predicted:
                y_pred.append(u_mean)
                y_true.append(true_r)
                fallback_cnt += 1

        rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
        mae_val = mean_absolute_error(y_true, y_pred)
        avg_ms = (e_time / len(test_sample)) * 1000

        results.append({
            'seed': SEED,
            'rmse': rmse_val,
            'mae': mae_val,
            'avg_ms': avg_ms,
            'fallback_count': fallback_cnt,
            'matrix_time': e_time
        })

        # Seed 10일 때 페르소나 추천 저장
        if SEED == 10:
            best_reco['heavy'] = get_recommendations(PERSONA_HEAVY_USER, pred_matrix_z, u_ids, m_ids, valid_mask, user_stats, train)
            best_reco['specialist'] = get_recommendations(PERSONA_GENRE_SPECIALIST, pred_matrix_z, u_ids, m_ids, valid_mask, user_stats, train)

    # --- [상세 출력: 요청하신 포맷] ---
    df_res = pd.DataFrame(results)

    print("\n=== 실험 결과 요약 ===")
    print(df_res[['seed', 'rmse', 'mae', 'avg_ms', 'fallback_count']].to_string(index=False))

    print("\n=== 평균 ± 표준편차 ===")
    print(f"RMSE : {df_res['rmse'].mean():.4f} ± {df_res['rmse'].std():.4f}")
    print(f"MAE  : {df_res['mae'].mean():.4f} ± {df_res['mae'].std():.4f}")
    print(f"폴백 사용 케이스 : {df_res['fallback_count'].mean():.1f}개 (± {df_res['fallback_count'].std():.1f})")
    print(f"예측 속도 (샘플 기준) : {df_res['avg_ms'].mean():.2f} ms (± {df_res['avg_ms'].std():.2f})")
    print(f"행렬 계산 시간 : {df_res['matrix_time'].mean():.4f} s (± {df_res['matrix_time'].std():.4f})")

    print("\n=== 페르소나 추천 결과 (Seed 10) ===")
    print_reco("헤비 유저", PERSONA_HEAVY_USER, best_reco.get('heavy'))
    print_reco("드라마 전문가", PERSONA_GENRE_SPECIALIST, best_reco.get('specialist'))

    # 글로벌 요약 저장을 위해 기록
    global_summary.append({
        'Shrinkage': shrinkage_val,
        'Avg_RMSE': df_res['rmse'].mean(),
        'Avg_MAE': df_res['mae'].mean(),
        'Avg_Time': df_res['matrix_time'].mean()
    })

# ---------------------------------------------------------
# 6. 최종 비교 (전체 요약)
# ---------------------------------------------------------
print(f"\n{'='*60}")
print(" [최종 종합 비교] Shrinkage 파라미터별 성능")
print(f"{'='*60}")
df_global = pd.DataFrame(global_summary)
print(df_global)

# 최적 파라미터 출력
best_row = df_global.loc[df_global['Avg_RMSE'].idxmin()]
print(f"\nBest Shrinkage Parameter: {int(best_row['Shrinkage'])}")
print(f"   (Best RMSE: {best_row['Avg_RMSE']:.4f})")


 실험 파라미터: Shrinkage = 0
 (0이면 미적용, 숫자가 클수록 신뢰도 필터링 강도 높음)

=== 실험 결과 요약 ===
 seed     rmse      mae   avg_ms  fallback_count
   10 0.856606 0.648962 3.012529             126
   21 0.863884 0.658738 3.421010             117
   35 0.868862 0.664197 3.498805             109
   42 0.882900 0.669712 2.885286             117
   57 0.836065 0.639204 3.354869             121

=== 평균 ± 표준편차 ===
RMSE : 0.8617 ± 0.0172
MAE  : 0.6562 ± 0.0122
폴백 사용 케이스 : 118.0개 (± 6.2)
예측 속도 (샘플 기준) : 3.23 ms (± 0.27)
행렬 계산 시간 : 9.7035 s (± 0.8083)

=== 페르소나 추천 결과 (Seed 10) ===
헤비 유저 (userId=414):
  - Harmonists, The (1997) (예측점수: 5.00)
  - Taxi 3 (2003) (예측점수: 4.93)
  - Spiral (2018) (예측점수: 4.75)
드라마 전문가 (userId=85):
  - Mr. Wrong (1996) (예측점수: 5.00)
  - Before and After (1996) (예측점수: 5.00)
  - Awfully Big Adventure, An (1995) (예측점수: 5.00)

 실험 파라미터: Shrinkage = 10
 (0이면 미적용, 숫자가 클수록 신뢰도 필터링 강도 높음)

=== 실험 결과 요약 ===
 seed     rmse      mae   avg_ms  fallback_count
   10 0.853374 0.644804 4.596907             126